# Tracker Smoke Test

Verify that ByteTrack assigns persistent IDs across consecutive frames.
Diagnostic only: tests the integration before moving to calibration (Stage 2).

In [1]:
import sys
sys.path.append("..")

import cv2
from pipeline.detection.tracker import Tracker

%load_ext autoreload
%autoreload 2

C:\Users\rohan\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tracker = Tracker("../pipeline/detection/football_yolo26n_best.pt", conf=0.1)
src = r"C:\Users\rohan\Desktop\Quant Sports Project\Tester video\08fd33_4.mp4"

cap = cv2.VideoCapture(src)
assert cap.isOpened(), f"Failed to open {src}"

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {fps} fps, {total_frames} frames")

Video: 25.0 fps, 750 frames


C:\Users\rohan\Desktop\Quant Sports Project\experiments\..\pipeline\detection\tracker.py:34: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  self.tracker = sv.ByteTrack()


In [3]:
frame_count = 0
max_frames = 100
id_history = {}

while frame_count < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    
    tracks, _ = tracker.track_frame(frame)
    for key, values in tracks.data.items():  # every .data array must match the boxes
        assert len(values) == len(tracks), f"{key} misaligned on frame {frame_count}"
    
    if tracks.tracker_id is not None:
        ids = sorted(set(int(id_) for id_ in tracks.tracker_id))
        id_history[frame_count] = ids
        
        if frame_count % 20 == 0:
            print(f"Frame {frame_count:3d}: {len(tracks)} objects, IDs: {ids}")
    else:
        print(f"Frame {frame_count:3d}: no tracks")
    
    frame_count += 1

cap.release()
print(f"\nProcessed {frame_count} frames")

Frame   0: 24 objects, IDs: [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


Frame  20: 23 objects, IDs: [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 24]


Frame  40: 24 objects, IDs: [-1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25]


Frame  60: 23 objects, IDs: [-1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 22, 24, 25, 26]


Frame  80: 23 objects, IDs: [-1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 22, 24, 25, 26]



Processed 100 frames


In [4]:
# Track continuity check.
# NOTE: this cannot measure ID switches (one player's ID jumping to another player).
# That needs labeled ground truth, i.e. HOTA on SoccerNet-Tracking, or the manual
# spot-check. What it does show is fragmentation: how often ByteTrack loses a track
# and issues a fresh ID for the same person.

BALL_ID = -1  # the ball is not a ByteTrack track, so leave it out of these stats

lifetimes = {}
for frame_num, ids in id_history.items():
    for id_ in ids:
        if id_ == BALL_ID:
            continue
        lifetimes[id_] = lifetimes.get(id_, 0) + 1

n_frames = len(id_history)
first_frame_ids = set(id_history[min(id_history)]) - {BALL_ID} if id_history else set()
new_ids = [i for i in lifetimes if i not in first_frame_ids]
fragments = [i for i, n in lifetimes.items() if n < 5]

print(f"Frames analysed:            {n_frames}")
print(f"People on frame 0:          {len(first_frame_ids)}")
print(f"Total unique IDs issued:    {len(lifetimes)}")
print(f"New IDs after frame 0:      {len(new_ids)}  (each one is a track that was lost or arrived)")
print(f"Short tracks (<5 frames):   {len(fragments)}")
print(f"Median track lifetime:      {sorted(lifetimes.values())[len(lifetimes) // 2]} frames")
print(f"IDs lasting all {n_frames} frames: {sum(1 for n in lifetimes.values() if n == n_frames)}")

# A perfect tracker on a fixed camera would issue about 22 to 25 IDs and keep them.
# Many extra IDs means fragmentation, which will show up later as broken player
# trajectories and unstable team assignment.

Frames analysed:            100
People on frame 0:          23
Total unique IDs issued:    26
New IDs after frame 0:      3  (each one is a track that was lost or arrived)
Short tracks (<5 frames):   1
Median track lifetime:      100 frames
IDs lasting all 100 frames: 17


In [5]:
import os
from pipeline.common.drawing import annotate_tracks

tracker.reset()  # fresh ByteTrack + ball state, since we rewind the video

cap = cv2.VideoCapture(src)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

os.makedirs("runs", exist_ok=True)
out = cv2.VideoWriter("runs/tracker_visual_check.avi", cv2.VideoWriter_fourcc(*"XVID"), fps, (w, h))

n_interp = 0
N_VISUAL = 300  # the full tuning range, frames 0 to 299
for i in range(N_VISUAL):
    ok, frame = cap.read()
    if not ok:
        break

    tracks, _ = tracker.track_frame(frame)
    n_interp += int(tracks.data["interpolated"].sum())
    annotated = annotate_tracks(frame, tracks, tracker.model.names)
    out.write(annotated)

cap.release()
out.release()
print(f"Saved to runs/tracker_visual_check.avi ({n_interp} predicted ball frames of {N_VISUAL})")
print("Filled ball triangle = detected. Outline triangle plus circle = Kalman prediction.")
print("The circle is 2 standard deviations wide, so it grows the longer the ball is unseen.")

Saved to runs/tracker_visual_check.avi (42 predicted ball frames of 300)
Filled ball triangle = detected. Outline triangle plus circle = Kalman prediction.
The circle is 2 standard deviations wide, so it grows the longer the ball is unseen.


In [6]:
import time
import numpy as np

tracker.reset()
cap = cv2.VideoCapture(src)
times = []

for i in range(51):
    ok, frame = cap.read()
    if not ok:
        break

    start = time.perf_counter()
    tracks, _ = tracker.track_frame(frame)
    elapsed = time.perf_counter() - start
    if i > 0:  # drop frame 0: it includes model warmup, not steady-state cost
        times.append(elapsed * 1000)

cap.release()

print(f"Mean: {np.mean(times):.2f}ms   (budget is 40ms at 25fps)")
print(f"P50:  {np.percentile(times, 50):.2f}ms")
print(f"P95:  {np.percentile(times, 95):.2f}ms")
print(f"Max:  {np.max(times):.2f}ms")

Mean: 53.93ms   (budget is 40ms at 25fps)
P50:  51.83ms
P95:  65.90ms
Max:  81.83ms
